In [1]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

In [2]:
augs = pl.scan_csv("augmenter_setups_4_null.csv").filter(
    pl.all_horizontal(pl.col("Damage", "Rate of Fire", "Shield Max").is_not_null()),
    (pl.col("Electric Tempering") <= 0.0) | (pl.col("Electric Tempering").is_null()),
    ~pl.col("").str.contains("Art of Commanding"),
    ~pl.col("").str.contains("Art of Destruction"),
    ~pl.col("").str.contains("Art of Fast Killing"),
    ~pl.col("").str.contains("Art of Gunnery"),
    ~pl.col("").str.contains("Art of Healing"),
    ~pl.col("").str.contains("Art of Sharpshooting"),
    ~pl.col("").str.contains("Art of Stealth"),
).collect()
# zeros already replaced with nan

# drop class locked augs
# for aug in ['Ultimate Art of Gunnery']:
#     mask = augs.index.str.contains(aug)
#     augs = augs.loc[~mask]


In [8]:
augs = augs.with_columns(
    pl.col("Transference Power").fill_null(strategy='zero'),
    pl.col("Critical Hit Strength").fill_null(strategy='zero'),
    pl.col("Critical Hit Chance").fill_null(strategy='zero')
)

augs = augs.with_columns(
    dps=(100 * (1 + pl.col("Damage")) * (1 + pl.col("Rate of Fire"))) * (1 + ((0.05 + pl.col("Critical Hit Chance")) * (0.5 * (1 + pl.col("Critical Hit Strength"))))),
    hps=(100 * (1 + pl.col("Transference Power")) * (1 + pl.col("Rate of Fire"))) * (1 + ((0.05 + pl.col("Critical Hit Chance")) * (0.5 * (1 + pl.col("Critical Hit Strength"))))),
)

# Old/original

In [4]:
interest = ('dps', 'Shield Max', 'Shield Recovery', 'Energy Max', 'Energy Charge', 'Resistance to Damage')

In [5]:
args = (pl.col(""), pl.col(*(i for i in interest if i not in ['Electric Tempering'])).rank(descending=True))
if 'Electric Tempering' in interest:
    args += (pl.col("Electric Tempering").rank(descending=False),)

augs_rank = augs.select(*args)

# fill with 2x size
augs_rank = augs_rank.select(
    pl.all().replace({None: 21e6 * 2})
)

In [6]:
augs_rank = augs_rank.with_columns(
    rank=pl.sum_horizontal(*interest)
)

In [7]:
augs = augs.with_columns(
    rank=augs_rank['rank']
)

In [8]:
with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000):
    display(augs.select(pl.col('', 'rank', *interest)).sort('rank', descending=False).head(15))

,rank,dps,Shield Max,Shield Recovery,Energy Max,Energy Charge,Resistance to Damage
str,f64,f64,f64,f64,f64,f64,f64
"""('Perilous Augmenter', ""Qa'ik Urk'qii Akk'oj"", 'Luminous Ossified Buttress Augmenter', 'Adamanturized Invigorating Augmenter')""",1.1332197e7,410.04,2.5,1.55,7.1,1.63,0.62
"""('Invigorating Dionysis Augmenter', 'Perilous Augmenter', ""Qa'ik Urk'qii Akk'oj"", 'Luminous Ossified Buttress Augmenter')""",1.1444e7,459.0,3.5,1.55,3.2,1.45,0.62
"""('Vaidya Bhava', 'Phased Fortification Augmenter', 'Adamanturized Defensive Augmenter', 'Adamanturized Invigorating Augmenter')""",1.1479e7,169.7355,3.407463,1.5,5.9,1.68,0.6
"""('Perilous Augmenter', ""Qa'ik Urk'qii Akk'oj"", ""Poseidon's Defensive Mode Augmenter"", 'Adamanturized Invigorating Augmenter')""",1.1632e7,316.2,2.6,1.75,5.9,1.33,0.8
"""('Perilous Augmenter', ""Qa'ik Urk'qii Akk'oj"", 'Adamanturized Defensive Augmenter', 'Adamanturized Invigorating Augmenter')""",1.1768e7,316.2,4.2,1.35,5.9,1.33,0.67
…,…,…,…,…,…,…,…
"""('Perilous Augmenter', 'Celestial Rage Augmenter', 'Luminous Ossified Buttress Augmenter', 'Luminous Ossified Buttress Augmenter')""",1.2410e7,576.24,3.7,2.0,2.4,1.05,0.67
"""('Vacohara Psu', 'Celestial Forge Augmenter', 'Luminous Brace Levee Augmenter', 'Luminous Ossified Buttress Augmenter')""",1.2423e7,344.034,3.85,1.8,3.381818,1.1,0.7
"""('Perilous Augmenter', 'Caraatiguru Psu', 'Luminous Brace Levee Augmenter', 'Luminous Ossified Buttress Augmenter')""",1.2484e7,534.29,2.5,1.8,4.2,1.55,0.46


In [13]:
augs.filter(
    augs[''].str.contains("Adamanturized Rage") & augs[''].str.contains("The Emperor's Augmenter") & augs[''].str.contains("Vaidya Bhava") & augs[''].str.contains("Nightfury's Patience")
)#.select(pl.col("", "rank", *interest, 'Electric Tempering', 'Transference Power'))

,Capacity,Critical Hit Chance,Critical Hit Strength,Damage,Docking Speed,Electric Tempering,Energy Charge,Energy Max,Hostility,Inertial Dampening,Multifiring,Projectile Velocity,Radar,Range,Rate of Fire,Resistance to Damage,Shield Max,Shield Recovery,Speed,Thrust,Tracking,Tractor Power,Tractor Range,Transference Power,Turning,Visibility,Weapon Hold,Weapons Slots,Weight,dps
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""(""Nightfury's Patience"", 'Vaid…",-0.1,0.1,0.3,1.8,null,-0.317076,0.25,0.45,null,null,null,null,null,0.8,3.0,0.45,1.5,1.7,null,-0.18,0.4,null,null,0.5,-0.33,2.0,null,null,null,1229.2


In [12]:
augs.filter(
    augs[''].str.contains("Ares Offensive") & augs[''].str.contains("The Emperor's Augmenter") & augs[''].str.contains("Celestial Rage") & augs[''].str.contains("Celestial Nebulae")
)#.select(pl.col("", "rank", *interest, 'Electric Tempering', 'Transference Power'))

,Capacity,Critical Hit Chance,Critical Hit Strength,Damage,Docking Speed,Electric Tempering,Energy Charge,Energy Max,Hostility,Inertial Dampening,Multifiring,Projectile Velocity,Radar,Range,Rate of Fire,Resistance to Damage,Shield Max,Shield Recovery,Speed,Thrust,Tracking,Tractor Power,Tractor Range,Transference Power,Turning,Visibility,Weapon Hold,Weapons Slots,Weight,dps
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""('Ares Offensive Mode Augmente…",null,0.58,1.2,1.31,null,-0.26,0.25,2.45,null,null,null,null,null,0.166667,2.72,0.75,1.1,null,0.71,null,0.24,null,null,null,0.22,-0.37,null,null,null,1454.82876


In [11]:
a1 = '''"Qa'ik Urk'qii Akk'oj"'''
a2 = "'Ares Offensive Mode Augmenter'"
a3 = "'Selenite Augmenter'"
augs.filter(
    pl.col("").str.contains(f"{a1}, {a1}, {a3}, {a2}")
)

,Capacity,Critical Hit Chance,Critical Hit Strength,Damage,Docking Speed,Electric Tempering,Energy Charge,Energy Max,Hostility,Inertial Dampening,Multifiring,Projectile Velocity,Radar,Range,Rate of Fire,Resistance to Damage,Shield Max,Shield Recovery,Speed,Thrust,Tracking,Tractor Power,Tractor Range,Transference Power,Turning,Visibility,Weapon Hold,Weapons Slots,Weight,dps
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""(""Qa'ik Urk'qii Akk'oj"", ""Qa'i…",null,0.48,1.6,1.61,null,-0.38,null,2.0,null,null,null,null,null,0.7,2.17,0.7,1.0,1.1,0.35,null,0.84,null,null,null,0.22,null,null,null,null,1397.42793


In [10]:
interest = ('dps', 'Damage', 'Rate of Fire', 'Critical Hit Chance', 'Critical Hit Strength', 'Shield Max', 'Shield Recovery', 'Energy Max', 'Energy Charge', 'Resistance to Damage')

with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000):
    display(augs.drop_nulls("dps").select(pl.col('', *interest)).sort('dps', descending=True).head(15))

,dps,Damage,Rate of Fire,Critical Hit Chance,Critical Hit Strength,Shield Max,Shield Recovery,Energy Max,Energy Charge,Resistance to Damage
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""(""Qa'ik Vazuk Akk'oj"", 'Empyreal Rage Augmenter', 'Empyreal Rage Augmenter', 'Luminous Obliterative Integrity Augmenter')""",2736.62256,2.79,5.72,0.07,0.7,1.5,null,null,null,0.15
"""('Celestial Burst Augmenter', 'Empyreal Rage Augmenter', 'Empyreal Rage Augmenter', 'Luminous Obliterative Integrity Augmenter')""",2697.23139,2.91,5.42,0.07,0.7,0.5,null,null,null,0.15
"""(""Qa'ik Vazuk Akk'oj"", 'Empyreal Rage Augmenter', 'Luminous Obliterative Integrity Augmenter', 'Adamanturized Rage Augmenter')""",2696.75861,2.94,5.37,0.07,0.7,1.5,null,null,null,0.15
"""('Empyreal Rage Augmenter', 'Total Eclipse Illunis Augmenter', 'Lustrous Mass Despoliation Augmenter', 'Lustrous Mass Despoliation Augmenter')""",2692.895625,3.05,4.95,0.1,1.35,0.5,0.25,1.0,null,-0.14
"""('Empyreal Rage Augmenter', 'Empyreal Rage Augmenter', 'Total Eclipse Illunis Augmenter', 'Lustrous Mass Despoliation Augmenter')""",2688.984375,2.85,5.25,0.1,1.35,0.5,0.25,1.0,null,-0.14
…,…,…,…,…,…,…,…,…,…,…
"""('Empyreal Rage Augmenter', 'Empyreal Rage Augmenter', 'Total Eclipse Illunis Augmenter', 'Adamanturized Rage Augmenter')""",2632.83,2.8,5.2,0.1,1.35,0.5,0.25,1.0,null,-0.14
"""('Total Eclipse Illunis Augmenter', 'Lustrous Mass Despoliation Augmenter', 'Lustrous Mass Despoliation Augmenter', 'Adamanturized Rage Augmenter')""",2628.36,3.2,4.6,0.1,1.35,0.5,0.25,1.0,null,-0.14
"""('Empyreal Rage Augmenter', 'Empyreal Rage Augmenter', 'Total Eclipse Illunis Augmenter', 'Luminous Obliterative Integrity Augmenter')""",2614.36139,2.94,4.42,0.17,2.05,0.5,0.25,1.0,null,-0.012629


In [ ]:
interest = ('dps', 'Damage', 'Rate of Fire', 'Critical Hit Chance', 'Critical Hit Strength', 'Shield Max', 'Shield Recovery', 'Energy Max', 'Energy Charge', 'Resistance to Damage')

with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000):
    display(augs.drop_nulls("dps").select(pl.col('', *interest)).sort('dps', descending=True).head(15))

In [3]:
augs.filter(
    pl.col("Critical Hit Chance") >= 0.45,
    pl.col("Critical Hit Strength") >= 1.5,
    pl.col("Damage") >= 1.5,
    pl.col("Rate of Fire") >= 2.0,
    pl.col("Shield Max") >= 1.0,
    pl.col("Resistance to Damage") >= 0.45,
    pl.col("Electric Tempering") <= -0.3,
    # pl.col("Energy Charge") >= 0.25,
    pl.col("Range") >= 0.2,
    # pl.col("Capacity") >= 0.0,
).drop(
    "Docking Speed",
    "Hostility",
    "Inertial Dampening",
    "Multifiring",
    "Projectile Velocity",
    "Radar",
    "Tractor Power",
    "Tractor Range",
    "Visibility",
    "Weapon Hold",
    "Weapons Slots"
)

,Capacity,Critical Hit Chance,Critical Hit Strength,Damage,Electric Tempering,Energy Charge,Energy Max,Range,Rate of Fire,Resistance to Damage,Shield Max,Shield Recovery,Speed,Thrust,Tracking,Transference Power,Turning,Weight
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""(""Nightfury's Anger"", 'Celesti…",0.018889,0.46,1.5,1.73,-0.300214,null,4.65,0.966667,2.3,0.75,1.4,null,0.51,-0.18,null,null,-0.33,null
"""('Raging Dionysis Augmenter', …",0.13,0.46,1.5,1.68,-0.400842,null,3.65,0.466667,2.2,0.75,1.0,null,0.36,-0.26,null,null,-0.1,null
"""(""Qa'ik Urk'qii Akk'oj"", ""Qa'i…",null,0.48,1.6,1.61,-0.38,null,2.0,0.7,2.17,0.7,1.0,1.1,0.35,null,0.84,null,0.22,null
"""(""Qa'ik Urk'qii Akk'oj"", ""Qa'i…",null,0.47,1.9,1.99,-0.38,null,null,0.7,2.02,0.85,1.0,1.1,null,null,0.6,null,null,null
"""(""Qa'ik Urk'qii Akk'oj"", 'Sele…",0.15,0.47,2.3,1.69,-0.486188,null,0.8,0.7,2.02,0.5,2.75,0.55,null,null,1.35,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""('Celestial Rage Augmenter', '…",0.15,0.58,1.8,2.0,-0.337662,null,0.8,0.316667,2.25,0.77,1.75,null,0.36,null,1.32,null,null,2.0
"""('Empyreal Rage Augmenter', 'L…",0.15,0.45,1.7,2.74,-0.337662,null,0.8,0.65,2.57,0.52,1.75,null,0.13,null,1.32,null,null,2.0
"""('Luminous Intersticial Desist…",0.41,0.51,1.7,2.44,-0.406977,null,2.1,0.8,2.02,0.47,1.75,null,null,null,0.75,null,null,null


In [4]:
augs.filter(
    pl.col("Critical Hit Chance") >= 0.45,
    pl.col("Critical Hit Strength") >= 1.5,
    pl.col("Damage") >= 1.5,
    pl.col("Rate of Fire") >= 2.0,
    pl.col("Shield Max") >= 1.0,
    pl.col("Resistance to Damage") >= 0.45,
    pl.col("Electric Tempering") <= -0.3,
    # pl.col("Energy Charge") >= 0.25,
    pl.col("Range") >= 0.2,
    # pl.col("Capacity") >= 0.0,
).write_excel("engineer_subset.xlsx")

In [10]:
with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000, tbl_rows=50):
    display(augs.filter(
        pl.col("dps") > 1600,
        pl.col("Range") > 0.0,
        pl.col("Shield Max") >= 1.0,
        pl.col("Shield Recovery") >= 1.0,
        pl.col("Resistance to Damage") >= 0.45,
        pl.col("Electric Tempering") <= -0.3,
    ).sort("dps", descending=True).drop(
        "Docking Speed",
        "Hostility",
        "Inertial Dampening",
        "Multifiring",
        "Projectile Velocity",
        "Tractor Power",
        "Tractor Range",
        "Visibility",
        "Weapon Hold",
        "Weapons Slots"
    ))

,Capacity,Critical Hit Chance,Critical Hit Strength,Damage,Electric Tempering,Energy Charge,Energy Max,Radar,Range,Rate of Fire,Resistance to Damage,Shield Max,Shield Recovery,Speed,Thrust,Tracking,Transference Power,Turning,Weight,dps
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""(""Nightfury's Patience"", 'Empyreal Rage Augmenter', 'Luminous Intersticial Desistance Augmenter', 'Luminous Resplendent Fluxion Augmenter')""",0.168889,0.32,1.0,1.9,-0.308568,null,1.45,null,0.7,3.3,0.46,1.75,1.0,0.13,-0.18,1.15,null,-0.33,null,1708.39
"""(""Qa'ik Urk'qii Akk'oj"", ""Qa'ik Urk'qii Akk'oj"", 'Selenite Augmenter', 'Empyreal Rage Augmenter')""",null,0.4,1.2,1.7,-0.441145,null,null,null,0.7,3.15,0.7,1.0,1.1,0.13,null,0.6,null,null,null,1675.1475
"""(""Qa'ik Urk'qii Akk'oj"", ""Qa'ik Urk'qii Akk'oj"", 'Selenite Augmenter', 'Lustrous Mass Despoliation Augmenter')""",null,0.4,1.2,1.9,-0.38,null,null,null,0.7,2.85,0.7,1.0,1.1,0.1,0.2,0.95,null,0.4,null,1669.1675
"""(""Nightfury's Patience"", 'Luminous Intersticial Desistance Augmenter', 'Luminous Resplendent Fluxion Augmenter', 'Adamanturized Rage Augmenter')""",0.168889,0.32,1.0,2.05,-0.308568,null,1.45,null,0.7,2.95,0.46,1.75,1.0,null,-0.18,1.15,null,-0.33,null,1650.5075
"""(""Qa'ik Urk'qii Akk'oj"", 'Celeste Mark Augmenter', 'Luminous Munitions Amplifier Augmenter', 'Luminous Obliterative Integrity Augmenter')""",0.1,0.27,1.6,2.7,-0.31,null,null,null,0.5,2.1,0.5,1.44,1.28,null,0.5,null,null,null,null,1624.152
"""(""Qa'ik Urk'qii Akk'oj"", ""Qa'ik Urk'qii Akk'oj"", 'Selenite Augmenter', 'Adamanturized Rage Augmenter')""",null,0.4,1.2,1.85,-0.441145,null,null,null,0.7,2.8,0.7,1.0,1.1,null,null,0.6,null,null,null,1619.085


In [13]:
augs.filter(
    pl.col("dps") > 1800,
    pl.col("Shield Max") >= 1.0,
    pl.col("Resistance to Damage") >= 0.45,
    pl.col("Electric Tempering") <= -0.3,
).write_excel("engineer_subset.xlsx")

In [10]:
with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000, tbl_rows=65):
    display(augs.filter(
        # ~pl.col("").str.contains("Luminous"),
        ~pl.col("").str.contains("Illunis"),
        ~pl.col("").str.contains("Promontory"),
        pl.col("dps") > 1500,
        pl.col("Shield Max") >= 1.0,
        pl.col("Resistance to Damage") >= 0.3,
        pl.col("Electric Tempering") <= -0.3,
        pl.col("Energy Charge") >= 0.25,
        pl.col("Range") > 0.6,
        pl.col("Shield Recovery") > 0.5
    ).sort("dps", descending=True).drop(
        "Docking Speed",
        "Hostility",
        "Inertial Dampening",
        "Multifiring",
        "Projectile Velocity",
        "Tractor Power",
        "Tractor Range",
        "Visibility",
        "Weapon Hold",
        "Weapons Slots"
    ))

,Capacity,Critical Hit Chance,Critical Hit Strength,Damage,Electric Tempering,Energy Charge,Energy Max,Radar,Range,Rate of Fire,Resistance to Damage,Shield Max,Shield Recovery,Speed,Thrust,Tracking,Transference Power,Turning,Weight,dps
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""(""Qa'ik Urk'qii Akk'oj"", 'Selenite Augmenter', 'Twisted Tesla Augmenter', 'Empyreal Rage Augmenter')""",null,0.3,0.6,2.05,-0.473819,0.6,0.75,null,0.7,3.45,0.35,1.0,0.55,0.13,null,0.6,null,null,null,1737.28
"""(""Qa'ik Urk'qii Akk'oj"", 'Selenite Augmenter', 'Twisted Tesla Augmenter', 'Lustrous Mass Despoliation Augmenter')""",null,0.3,0.6,2.25,-0.419958,0.6,0.75,null,0.7,3.15,0.35,1.0,0.55,0.1,0.2,0.95,null,0.4,null,1726.4
"""(""Qa'ik Urk'qii Akk'oj"", 'Celeste Mark Augmenter', 'Empyreal Rage Augmenter', ""The Emperor's Augmenter"")""",null,0.3,0.9,2.1,-0.345473,0.25,0.45,null,1.0,3.1,0.4,1.5,0.55,0.13,0.5,null,null,null,null,1693.6075
"""(""Qa'ik Urk'qii Akk'oj"", 'Selenite Augmenter', 'Twisted Tesla Augmenter', 'Adamanturized Rage Augmenter')""",null,0.3,0.6,2.2,-0.473819,0.6,0.75,null,0.7,3.1,0.35,1.0,0.55,null,null,0.6,null,null,null,1679.36
"""('Lunarian Augmenter', ""Qa'ik Urk'qii Akk'oj"", 'Selenite Augmenter', 'Lustrous Mass Despoliation Augmenter')""",null,0.2,0.6,1.95,-0.428332,0.5,null,null,0.7,3.65,0.35,1.65,0.55,0.1,0.2,0.95,null,0.4,null,1646.1
"""(""Qa'ik Urk'qii Akk'oj"", 'Selenite Augmenter', 'Twisted Tesla Augmenter', 'Luminous Obliterative Integrity Augmenter')""",null,0.37,1.3,2.34,-0.419958,0.6,0.75,null,0.7,2.32,0.5,1.0,0.55,null,null,0.6,null,null,null,1644.46904
"""('Lunarian Augmenter', ""Qa'ik Urk'qii Akk'oj"", 'Selenite Augmenter', 'Empyreal Rage Augmenter')""",null,0.2,0.6,1.75,-0.480718,0.5,null,null,0.7,3.95,0.35,1.65,0.55,0.13,null,0.6,null,null,null,1633.5
"""(""Qa'ik Urk'qii Akk'oj"", 'Celeste Mark Augmenter', ""The Emperor's Augmenter"", 'Adamanturized Rage Augmenter')""",null,0.3,0.9,2.25,-0.345473,0.25,0.45,null,1.0,2.75,0.4,1.5,0.55,null,0.5,null,null,null,null,1623.984375
"""(""Qa'ik Banu Akk'oj"", 'Luminous Munitions Amplifier Augmenter', 'Luminous Obliterative Integrity Augmenter', 'Luminous Obliterative Integrity Augmenter')""",0.19,0.14,1.4,2.74,-0.31,0.3,null,null,1.0,2.52,0.3,1.04,1.33,null,null,null,0.1,null,null,1616.63744


In [17]:
with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000, tbl_rows=50):
    display(augs.filter(
        pl.col("dps") > 1500,
        pl.col("Range") > 0.5,
        pl.col("Shield Max") >= 1.0,
        pl.col("Shield Recovery") >= 1.0,
        pl.col("Resistance to Damage") >= 0.45,
        (pl.col("Electric Tempering") <= -0.3) | ((pl.col("Electric Tempering") > 0) & (pl.col("Energy Charge") > 0.5)),
        pl.col("Energy Max") > 1
    ).sort("dps", descending=True).drop(
        "Docking Speed",
        "Hostility",
        "Inertial Dampening",
        "Multifiring",
        "Projectile Velocity",
        "Tractor Power",
        "Tractor Range",
        "Visibility",
        "Weapon Hold",
        "Weapons Slots"
    ))

,Capacity,Critical Hit Chance,Critical Hit Strength,Damage,Electric Tempering,Energy Charge,Energy Max,Radar,Range,Rate of Fire,Resistance to Damage,Shield Max,Shield Recovery,Speed,Thrust,Tracking,Transference Power,Turning,Weight,dps
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""(""Nightfury's Patience"", 'Empyreal Rage Augmenter', 'Luminous Intersticial Desistance Augmenter', 'Luminous Resplendent Fluxion Augmenter')""",0.168889,0.32,1.0,1.9,-0.308568,null,1.45,null,0.7,3.3,0.46,1.75,1.0,0.13,-0.18,1.15,null,-0.33,null,1708.39
"""(""Nightfury's Patience"", 'Luminous Intersticial Desistance Augmenter', 'Luminous Resplendent Fluxion Augmenter', 'Adamanturized Rage Augmenter')""",0.168889,0.32,1.0,2.05,-0.308568,null,1.45,null,0.7,2.95,0.46,1.75,1.0,null,-0.18,1.15,null,-0.33,null,1650.5075
"""(""Nightfury's Patience"", 'Raging Dionysis Augmenter', 'Empyreal Rage Augmenter', 'Luminous Brace Levee Augmenter')""",-0.1,0.14,0.7,2.03,-0.300214,null,3.0,null,0.7,3.45,0.49,1.0,1.0,0.13,-0.363407,0.4,null,-0.376422,null,1566.108525
"""(""Nightfury's Patience"", 'Celeste Mark Augmenter', 'Empyreal Rage Augmenter', 'Luminous Brace Levee Augmenter')""",-0.1,0.14,1.0,2.03,-0.300214,null,3.0,null,1.2,3.3,0.49,1.0,1.0,0.13,0.280488,0.4,null,-0.33,null,1550.451
"""(""Nightfury's Patience"", 'Raging Dionysis Augmenter', 'Luminous Brace Levee Augmenter', 'Adamanturized Rage Augmenter')""",-0.1,0.14,0.7,2.18,-0.300214,null,3.0,null,0.7,3.1,0.49,1.0,1.0,null,-0.363407,0.4,null,-0.376422,null,1514.3637


# New/hps

In [11]:
tmp = augs.filter(
    pl.col("Shield Max") >= 2.0,
    pl.col("Shield Recovery") >= 2.5,
    pl.col("Resistance to Damage") >= 0.3,
    # pl.col("Electric Tempering") <= -0.1,
    pl.col("Range") >= 0.5,
    pl.col("hps") >= 800
).drop(
    "Docking Speed",
    "Hostility",
    "Inertial Dampening",
    "Multifiring",
    "Projectile Velocity",
    "Radar",
    "Tractor Power",
    "Tractor Range",
    "Visibility",
    "Weapon Hold",
    "Weapons Slots"
)

with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000, tbl_rows=20):
    display(tmp.sort("hps", descending=True))

,Capacity,Critical Hit Chance,Critical Hit Strength,Damage,Electric Tempering,Energy Charge,Energy Max,Range,Rate of Fire,Resistance to Damage,Shield Max,Shield Recovery,Speed,Thrust,Tracking,Transference Power,Turning,Weight,dps,hps
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""(""Poseidon's Defensive Mode Augmenter"", ""Poseidon's Defensive Mode Augmenter"", 'Celestial Enduring Augmenter', 'Empyreal Rage Augmenter')""",null,0.0,0.0,0.6,-0.15,null,1.0,0.5,2.6,0.36,2.9,3.4,0.51,0.2,null,1.6,null,null,590.4,959.4
"""(""Qa'ik Banu Akk'oj"", ""Poseidon's Defensive Mode Augmenter"", ""Poseidon's Defensive Mode Augmenter"", 'Empyreal Rage Augmenter')""",0.09,0.0,0.0,0.95,-0.15,0.3,null,1.0,2.2,0.36,3.2,3.0,0.13,null,null,1.7,null,null,639.6,885.6
"""(""Poseidon's Defensive Mode Augmenter"", ""Poseidon's Defensive Mode Augmenter"", 'Celestial Enduring Augmenter', 'Lustrous Mass Despoliation Augmenter')""",null,0.0,0.0,0.8,null,null,1.0,0.5,2.3,0.36,2.9,3.4,0.48,0.4,0.35,1.6,0.4,null,608.85,879.45
"""(""Poseidon's Defensive Mode Augmenter"", ""Poseidon's Defensive Mode Augmenter"", 'Celestial Enduring Augmenter', 'Adamanturized Rage Augmenter')""",null,0.0,0.0,0.75,-0.15,null,1.0,0.5,2.25,0.36,2.9,3.4,0.38,0.2,null,1.6,null,null,582.96875,866.125
"""(""Poseidon's Defensive Mode Augmenter"", 'Celestial Enduring Augmenter', 'Empyreal Rage Augmenter', 'Luminous Vitreous Stimulative Augmenter')""",null,0.0,0.0,1.27,-0.15,null,1.0,0.5,2.6,0.37,2.3,2.8,0.51,0.2,0.5,1.33,null,null,837.63,859.77
"""(""Poseidon's Defensive Mode Augmenter"", ""Poseidon's Defensive Mode Augmenter"", 'Empyreal Ascension Augmenter', 'Empyreal Rage Augmenter')""",-0.25,0.0,0.0,0.6,-0.15,0.46,2.0,0.85,1.85,0.36,2.6,2.9,0.13,-0.16,0.5,1.9,-0.3,null,467.4,847.1625
"""(""Poseidon's Defensive Mode Augmenter"", 'Celestial Enduring Augmenter', 'Empyreal Rage Augmenter', 'Gilded Necropolis Augmenter')""",0.1,0.0,0.0,0.6,-0.025788,null,2.0,0.5,2.6,0.73,2.15,2.7,0.64,0.2,null,1.25,-0.33,null,590.4,830.25
"""(""Nightfury's Patience"", 'Selenite Augmenter', ""Poseidon's Defensive Mode Augmenter"", ""Poseidon's Defensive Mode Augmenter"")""",-0.1,0.0,0.0,0.4,-0.271617,null,null,1.0,2.1,0.66,3.6,3.4,null,-0.18,1.0,1.6,-0.33,null,444.85,826.15
"""(""Nightfury's Patience"", 'Selenite Augmenter', ""Poseidon's Defensive Mode Augmenter"", 'Dhatriatiguru Bhava')""",-0.1,0.0,0.0,0.4,-0.271617,0.75,null,1.4,2.1,0.48,2.3,2.95,null,-0.18,1.0,1.6,-0.33,null,444.85,826.15
